# Plot Analyses And Results

This notebook demonstrates retrieval and plotting through the high-level results service.

It uses `pyecharts` through `ResultsService.plot_results(...)` and can run entirely with in-memory example data.

In [7]:
from __future__ import annotations

import enum
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyecharts.globals import CurrentConfig, NotebookType

if not hasattr(enum, "StrEnum"):
    class _CompatStrEnum(str, enum.Enum):
        pass
    enum.StrEnum = _CompatStrEnum

CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_NOTEBOOK

workspace_root = Path.cwd().resolve().parent
sys.path.insert(0, str(workspace_root / "src"))
sys.path.insert(0, str(workspace_root.parent / "owi-metadatabase-sdk" / "src"))

from owi.metadatabase.results import (
    ResultsService,
    WindSpeedHistogram,
    plot_ceit_analyses,
    plot_lifetime_design_frequencies_by_location,
    plot_lifetime_design_frequencies_geo,
)
from owi.metadatabase.results.analyses.lifetime_design_frequencies import LifetimeDesignFrequencies
from owi.metadatabase.results.analyses.lifetime_design_verification import LifetimeDesignVerification


class StubRepository:
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame

    def list_results(self, query):
        return self.frame

    def create_analysis(self, payload):
        return payload

    def create_results_bulk(self, payloads):
        return {"items": list(payloads)}


## Histogram Example

In [3]:
histogram_analysis = WindSpeedHistogram()
histogram_series = histogram_analysis.to_results(
    {
        "series": [
            {
                "title": "Design",
                "scope_label": "Site",
                "site_id": 1,
                "bins": [(0.0, 1.0), (1.0, 2.0), (2.0, 3.0)],
                "values": [4.0, 7.0, 3.0],
            },
            {
                "title": "Measured",
                "scope_label": "Site",
                "site_id": 1,
                "bins": [(0.0, 1.0), (1.0, 2.0), (2.0, 3.0)],
                "values": [3.0, 8.0, 4.0],
            },
        ]
    }
)
histogram_frame = pd.DataFrame([item.to_record_payload(analysis_id=101) for item in histogram_series])
histogram_service = ResultsService(repository=StubRepository(histogram_frame))
histogram_data = histogram_service.get_results("WindSpeedHistogram")
histogram_plot = histogram_service.plot_results("WindSpeedHistogram")

display(histogram_data)
display(histogram_plot.chart.render_notebook())


,series_name,scope,bin_left,bin_right,value
0,Design,Site,0.0,1.0,4.0
1,Design,Site,1.0,2.0,7.0
2,Design,Site,2.0,3.0,3.0
3,Measured,Site,0.0,1.0,3.0
4,Measured,Site,1.0,2.0,8.0
5,Measured,Site,2.0,3.0,4.0


## Time-Series Example

In [4]:
verification_analysis = LifetimeDesignVerification()
verification_series = verification_analysis.to_results(
    {
        "rows": [
            {
                "timestamp": datetime(2024, 1, 1, tzinfo=timezone.utc),
                "turbine": "BBA01",
                "FA1": 0.356,
                "SS1": 0.357,
                "location_id": 5,
            },
            {
                "timestamp": datetime(2024, 1, 2, tzinfo=timezone.utc),
                "turbine": "BBA01",
                "FA1": 0.355,
                "SS1": 0.356,
                "location_id": 5,
            },
        ]
    }
)
verification_frame = pd.DataFrame([item.to_record_payload(analysis_id=102) for item in verification_series])
verification_service = ResultsService(repository=StubRepository(verification_frame))
verification_data = verification_service.get_results("LifetimeDesignVerification")
verification_plot = verification_service.plot_results("LifetimeDesignVerification")

display(verification_data)
verification_plot.chart.render_notebook()


,x,y,series_name,turbine,metric
0,2024-01-01T00:00:00+00:00,0.356,BBA01 - FA1,BBA01,FA1
1,2024-01-02T00:00:00+00:00,0.355,BBA01 - FA1,BBA01,FA1
2,2024-01-01T00:00:00+00:00,0.357,BBA01 - SS1,BBA01,SS1
3,2024-01-02T00:00:00+00:00,0.356,BBA01 - SS1,BBA01,SS1


## Comparison Example

In [ ]:
frequencies_analysis = LifetimeDesignFrequencies()
frequencies_series = frequencies_analysis.to_results(
    {
        "rows": [
            {
                "turbine": "BBA01",
                "reference": "INFL",
                "FA1": 0.3406,
                "SS1": 0.3407,
                "location_id": 9,
            },
            {
                "turbine": "BBA01",
                "reference": "ACTU",
                "FA1": 0.3330,
                "SS1": 0.3332,
                "location_id": 9,
            },
        ]
    }
)
frequencies_frame = pd.DataFrame([item.to_record_payload(analysis_id=103) for item in frequencies_series])
frequencies_service = ResultsService(repository=StubRepository(frequencies_frame))
frequencies_data = frequencies_service.get_results("LifetimeDesignFrequencies")
frequencies_plot = frequencies_service.plot_results("LifetimeDesignFrequencies")

display(frequencies_data)
frequencies_plot.chart.render_notebook()


,x,y,series_name,turbine,metric,reference,location_id,site_id
0,INFL,0.3406,BBA01 - FA1,BBA01,FA1,INFL,9,None
1,ACTU,0.3330,BBA01 - FA1,BBA01,FA1,ACTU,9,None
2,INFL,0.3407,BBA01 - SS1,BBA01,SS1,INFL,9,None
3,ACTU,0.3332,BBA01 - SS1,BBA01,SS1,ACTU,9,None


## CEIT And Location-Aware Frequency Plots

The next cell demonstrates the new CEIT sensor dropdown plot plus the two lifetime design frequency dropdown plots that use location names and coordinates.

In [6]:
ceit_frame = pd.DataFrame(
    [
        {"sensor_identifier": "DA8F", "timestamp": "2025-03-12T12:05:12+00:00", "metric": "temperature", "value": 10.0},
        {"sensor_identifier": "DA8F", "timestamp": "2025-03-12T12:06:12+00:00", "metric": "temperature", "value": 11.0},
        {"sensor_identifier": "DA8F", "timestamp": "2025-03-12T12:05:12+00:00", "metric": "battery", "value": 4.1},
        {"sensor_identifier": "DA8F", "timestamp": "2025-03-12T12:06:12+00:00", "metric": "battery", "value": 4.0},
        {"sensor_identifier": "DA9D", "timestamp": "2025-03-12T12:05:12+00:00", "metric": "temperature", "value": 9.8},
        {"sensor_identifier": "DA9D", "timestamp": "2025-03-12T12:06:12+00:00", "metric": "temperature", "value": 9.9},
        {"sensor_identifier": "DA9D", "timestamp": "2025-03-12T12:05:12+00:00", "metric": "tof", "value": 1653.4},
        {"sensor_identifier": "DA9D", "timestamp": "2025-03-12T12:06:12+00:00", "metric": "tof", "value": 1653.5},
    ]
)
location_frame = pd.DataFrame(
    [
        {"id": 9, "title": "BBA01", "northing": 51.50, "easting": 2.80},
        {"id": 10, "title": "BBA02", "northing": 51.51, "easting": 2.82},
    ]
)
frequencies_dropdown_frame = pd.DataFrame(
    [
        {"location_id": 9, "turbine": "BBA01", "metric": "FA1", "reference": "INFL", "y": 0.3406},
        {"location_id": 9, "turbine": "BBA01", "metric": "SS1", "reference": "INFL", "y": 0.3407},
        {"location_id": 9, "turbine": "BBA01", "metric": "FA2", "reference": "ACTU", "y": 0.3330},
        {"location_id": 9, "turbine": "BBA01", "metric": "SS2", "reference": "ACTU", "y": 0.3332},
        {"location_id": 10, "turbine": "BBA02", "metric": "FA1", "reference": "INFL", "y": 0.3201},
        {"location_id": 10, "turbine": "BBA02", "metric": "SS1", "reference": "INFL", "y": 0.3204},
        {"location_id": 10, "turbine": "BBA02", "metric": "FA2", "reference": "ACTU", "y": 0.3111},
        {"location_id": 10, "turbine": "BBA02", "metric": "SS2", "reference": "ACTU", "y": 0.3114},
    ]
)

ceit_plot = plot_ceit_analyses(ceit_frame)
frequency_location_plot = plot_lifetime_design_frequencies_by_location(
    frequencies_dropdown_frame,
    location_frame=location_frame,
)
frequency_geo_plot = plot_lifetime_design_frequencies_geo(
    frequencies_dropdown_frame,
    location_frame=location_frame,
)

display(ceit_plot.notebook)
display(frequency_location_plot.notebook)
display(frequency_geo_plot.notebook)
